In [ ]:
# Thiết lập cho Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 7: Kỹ thuật Phân cụm (Clustering Techniques)

**Nhiệm vụ:**
1. **K-Means, Mean Shift, DBSCAN**: Phân khúc khách hàng (Customer Segmentation) bằng K-Means. So sánh với DBSCAN trên bộ dữ liệu có hình dạng bất thường (phi cầu).
2. **Mô hình Hỗn hợp Gauss (Gaussian Mixture Models - GMM)**: Phát hiện lỗi sản phẩm (Product Defect Detection) hoặc điểm bất thường sử dụng GMM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, MeanShift, DBSCAN, estimate_bandwidth
from sklearn.mixture import GaussianMixture

# Thiết lập style cho biểu đồ
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Phân khúc Khách hàng bằng K-Means

Chúng ta sẽ mô phỏng một tập dữ liệu khách hàng với các đặc trưng như 'Thu nhập hàng năm' (Annual Income) và 'Điểm chi tiêu' (Spending Score). Sau đó, sử dụng K-Means để xác định các nhóm khách hàng khác biệt.

In [ ]:
# Tạo dữ liệu khách hàng tổng hợp (ví dụ: 5 phân khúc dựa trên thu nhập và chi tiêu)
X_customers, y_customers = make_blobs(n_samples=300, centers=5, cluster_std=1.0, random_state=42)

# Áp dụng thuật toán K-Means
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
customer_segments = kmeans.fit_predict(X_customers)

# Trực quan hóa các phân khúc
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_customers[:, 0], X_customers[:, 1], c=customer_segments, cmap='viridis', s=50, alpha=0.7)
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.9, marker='X', label='Trọng tâm (Centroids)')
plt.title('Phân khúc Khách hàng với K-Means')
plt.xlabel('Thu nhập hàng năm (Đã chuẩn hóa)')
plt.ylabel('Điểm chi tiêu (Đã chuẩn hóa)')
plt.legend()
plt.show()

## 2. Phân cụm Mean Shift
Mean Shift là thuật toán dựa trên trọng tâm (centroid-based) phát hiện các cụm mà không cần khai báo trước số lượng cụm. Nó tìm kiếm các vùng dữ liệu dày đặc.

In [ ]:
# Ước tính băng thông (bandwidth) cho Mean Shift
bandwidth = estimate_bandwidth(X_customers, quantile=0.2, n_samples=300)

ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(X_customers)
ms_labels = ms.labels_
ms_centers = ms.cluster_centers_

plt.figure(figsize=(8, 5))
plt.scatter(X_customers[:, 0], X_customers[:, 1], c=ms_labels, cmap='plasma', s=50, alpha=0.7)
plt.scatter(ms_centers[:, 0], ms_centers[:, 1], c='red', s=200, alpha=0.9, marker='X', label='Trọng tâm (Centroids)')
plt.title(f'Phân cụm Mean Shift (Đã tìm thấy {len(ms_centers)} cụm)')
plt.xlabel('Thu nhập hàng năm (Đã chuẩn hóa)')
plt.ylabel('Điểm chi tiêu (Đã chuẩn hóa)')
plt.legend()
plt.show()

## 3. So sánh: K-Means vs DBSCAN trên Dữ liệu Hình dạng lạ
K-Means gặp khó khăn với các cụm không có hình dạng hình cầu. DBSCAN, một thuật toán dựa trên mật độ, có thể xác định các cụm với hình dáng bất kỳ và có tính chống nhiễu (outliers) tốt.

In [ ]:
# Tạo tập dữ liệu hình mặt trăng (không phải hình cầu)
X_moons, y_moons = make_moons(n_samples=500, noise=0.05, random_state=42)

# K-Means trên tập dữ liệu moons
kmeans_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans_moons.fit_predict(X_moons)

# DBSCAN trên tập dữ liệu moons
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

# Vẽ đồ thị so sánh
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_labels, cmap='viridis', s=50)
ax1.set_title('Phân cụm K-Means (Thất bại trong việc phân tách đúng)')

ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_labels, cmap='viridis', s=50)
ax2.set_title('Phân cụm DBSCAN (Thành công)')

plt.show()

## 4. Phát hiện Lỗi Sản phẩm (Anomaly Detection) bằng GMM
GMM có thể được sử dụng để phát hiện lỗi/điểm bất thường bằng cách ước lượng mật độ xác suất của dữ liệu. Những sản phẩm/vật thể rơi vào vùng có mật độ thấp (xác suất thấp) sẽ được đánh dấu là lỗi hoặc bất thường.

In [ ]:
# Tạo dữ liệu sản phẩm "bình thường" (normal) và thêm vào một số điểm "lỗi/bất thường" (anomalies)
np.random.seed(42)
X_normal, _ = make_blobs(n_samples=500, centers=2, cluster_std=1.5, random_state=42)
X_anomalies = np.random.uniform(low=-10, high=10, size=(20, 2))
X_gmm = np.vstack([X_normal, X_anomalies])

# Huấn luyện mô hình GMM
gmm = GaussianMixture(n_components=2, n_init=10, random_state=42)
gmm.fit(X_gmm)

# Tính toán mật độ (log xác suất) cho mỗi mẫu
densities = gmm.score_samples(X_gmm)

# Đặt ngưỡng để xác định lỗi/bất thường (ví dụ: 4% các điểm có mật độ thấp nhất)
density_threshold = np.percentile(densities, 4)
anomalies = X_gmm[densities < density_threshold]

# Vẽ đồ thị
plt.figure(figsize=(10, 6))
plt.scatter(X_gmm[:, 0], X_gmm[:, 1], c='blue', s=20, alpha=0.5, label='Sản phẩm bình thường')
plt.scatter(anomalies[:, 0], anomalies[:, 1], c='red', s=100, marker='o', edgecolors='black', label='Lỗi sản phẩm (Anomalies)')

plt.title('Phát hiện Lỗi Sản phẩm bằng GMM (Xác định các điểm có mật độ thấp)')
plt.legend()
plt.show()